In [ ]:
import os, sys, json, warnings, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix
from PIL import Image
import umap
warnings.filterwarnings("ignore")

BASE   = r"C:/Users/ACER/OneDrive/Documents/final-project"
RES    = os.path.join(BASE, "results")
CACHE  = os.path.join(BASE, "embeddings_cache")
IMGDIR = os.path.join(BASE, "data", "image-data")
DOCS   = os.path.join(BASE, "docs")
os.makedirs(DOCS, exist_ok=True)

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 150,
})
CMAP16 = plt.get_cmap("tab20")
CMAP10 = plt.get_cmap("tab10")
print("Setup OK")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8))
ax.set_xlim(0, 13); ax.set_ylim(0, 8); ax.axis("off")

def box(ax, x, y, w, h, text, fc="#E8F4FD", ec="#2C7BB6", fs=10):
    from matplotlib.patches import FancyBboxPatch
    rect = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1",
                          facecolor=fc, edgecolor=ec, linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x+w/2, y+h/2, text, ha="center", va="center", fontsize=fs,
            multialignment="center")

def arr(ax, x1, y1, x2, y2, style="->", col="#555"):
    ax.annotate("", xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle=style, color=col, lw=1.5))

box(ax, 0.3, 6.2, 2.4, 1.2, "Text Data\n4800 articles\n16 categories",   "#FFF9E6", "#E67E22")
box(ax, 3.3, 6.2, 2.4, 1.2, "Image Data\n5500 images\n10 categories",    "#FFF9E6", "#E67E22")
box(ax, 0.3, 4.4, 2.4, 1.2, "BART summary\n+\nSBERT\n(MiniLM/MPNet/RoBERTa)", "#EAF5EA", "#27AE60")
box(ax, 3.3, 4.4, 2.4, 1.2, "CLIP\nViT-B/32\n512d",                           "#EAF5EA", "#27AE60")
box(ax, 6.5, 5.8, 2.8, 1.4, "Unsupervised\nKMeans / Agglo / DEC",       "#EEE5FF", "#8E44AD")
box(ax, 6.5, 4.0, 2.8, 1.4, "Semi-supervised\nSnorkel / LP / Pseudo",   "#EEE5FF", "#8E44AD")
box(ax, 6.5, 2.2, 2.8, 1.4, "Supervised baseline\nDistilBERT / ELECTRA","#FFECEC", "#C0392B")
box(ax, 10.1, 4.2, 2.4, 1.2, "Evaluation\nACC / F1 / NMI\nARI / Silhouette", "#FFF3CD", "#F39C12")
box(ax, 10.1, 2.4, 2.4, 1.2, "Demo UI\nGradio",                                "#D5F5E3", "#1ABC9C")

arr(ax, 1.5, 6.2, 1.5, 5.6); arr(ax, 4.5, 6.2, 4.5, 5.6)
arr(ax, 2.7, 5.0, 6.5, 6.5); arr(ax, 2.7, 5.0, 6.5, 4.7)
arr(ax, 2.7, 5.0, 6.5, 2.9); arr(ax, 5.7, 5.0, 6.5, 6.5)
arr(ax, 9.3, 6.5, 10.1, 4.9); arr(ax, 9.3, 4.7, 10.1, 4.7)
arr(ax, 9.3, 2.9, 10.1, 4.5); arr(ax, 11.3, 4.2, 11.3, 3.6)

ax.text(1.5, 3.8, "5% seed\nshared", ha="center", va="top", fontsize=9,
        color="#E74C3C", style="italic")
ax.annotate("", xy=(5.5, 2.9), xytext=(5.5, 3.8),
            arrowprops=dict(arrowstyle="->", color="#E74C3C", lw=1.2, linestyle="dashed"))

ax.set_title("HINH 3.1: Kien truc tong the he thong pipeline sinh nhan tu dong", fontsize=12, pad=10)
plt.tight_layout()
plt.savefig(os.path.join(DOCS, "fig3_1_pipeline_architecture.png"), bbox_inches="tight", dpi=150)
plt.show(); print("fig3_1 saved")

In [ ]:
labels_df = pd.read_csv(os.path.join(RES, "full_labels_mpnet_kmeans.csv"))
all_labels = labels_df["true_label"].values
N = 500

embed_configs = [
    ("MiniLM-L6",    np.load(os.path.join(CACHE, "minilm_train_full_summary.npy"))[:N], all_labels[:N]),
    ("MPNet-base",   np.load(os.path.join(CACHE, "mpnet_train_full_summary.npy"))[:N],  all_labels[:N]),
    ("RoBERTa-large",np.load(os.path.join(CACHE, "roberta_train_n500.npy")),            all_labels[:500]),
]
cats = sorted(labels_df["true_label"].unique())

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (name, emb, lbs) in zip(axes, embed_configs):
    print(f"t-SNE {name} {emb.shape}...")
    tsne = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=500)
    xy = tsne.fit_transform(emb)
    for i, cat in enumerate(cats):
        mask = lbs == cat
        ax.scatter(xy[mask,0], xy[mask,1], c=[CMAP16(i/16)], s=12, alpha=0.7, linewidths=0, label=cat)
    ax.set_title(name, fontsize=11); ax.set_xticks([]); ax.set_yticks([])

handles = [mpatches.Patch(color=CMAP16(i/16), label=c) for i, c in enumerate(cats)]
fig.legend(handles=handles, loc="lower center", ncol=8, fontsize=8.5, frameon=False, bbox_to_anchor=(0.5, -0.05))
fig.suptitle("HINH 3.4: t-SNE cua ba khong gian nhung van ban (500 mau, to mau theo nhan thuc te)", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(DOCS, "fig3_4_tsne_text_embeddings.png"), bbox_inches="tight", dpi=150)
plt.show(); print("fig3_4 saved")

In [ ]:
from transformers import CLIPModel, CLIPProcessor
import torch

print("Loading CLIP model...")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model.eval()

classes = sorted(os.listdir(IMGDIR))
N_PER = 50
all_embeds, all_lbls = [], []
for cls in classes:
    imgs_paths = sorted(glob.glob(os.path.join(IMGDIR, cls, "*")))[:N_PER]
    batch = []
    for p in imgs_paths:
        try: batch.append(Image.open(p).convert("RGB"))
        except: pass
    if not batch: continue
    with torch.no_grad():
        inp = processor(images=batch, return_tensors="pt", padding=True)
        out = model.get_image_features(**inp)
        feats = out.pooler_output if hasattr(out, "pooler_output") else out
        feats = feats / feats.norm(dim=-1, keepdim=True)
    all_embeds.append(feats.numpy()); all_lbls.extend([cls]*len(batch))
    print(f"  {cls}: {len(batch)}")

X = np.vstack(all_embeds); lbs = np.array(all_lbls)
print(f"Total: {X.shape}")
print("t-SNE...")
xy = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=500).fit_transform(X)

fig, ax = plt.subplots(figsize=(9, 7))
for i, cls in enumerate(classes):
    mask = lbs == cls
    ax.scatter(xy[mask,0], xy[mask,1], c=[CMAP10(i/10)], s=18, alpha=0.8, label=cls, linewidths=0)
ax.legend(fontsize=9, frameon=False)
ax.set_xticks([]); ax.set_yticks([])
ax.set_title(f"HINH 3.5: t-SNE khong gian nhung CLIP ViT-B/32 — 10 lop dong vat ({N_PER} mau/lop)", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(DOCS, "fig3_5_tsne_clip_embeddings.png"), bbox_inches="tight", dpi=150)
plt.show(); print("fig3_5 saved")

In [ ]:
with open(os.path.join(RES, "image_cluster_descriptions_kmeans.json")) as f:
    descs = json.load(f)
with open(os.path.join(RES, "image_cluster_mapping_kmeans.json")) as f:
    mapping = json.load(f)

df_img = pd.read_csv(os.path.join(RES, "image_full_labels_unsup_kmeans.csv"))
examples = [("0","cow"),("1","spider"),("2","cat"),("4","chicken")]

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, (cid, cls) in zip(axes, examples):
    sub = df_img[(df_img["predicted_label"]==cls) & (df_img["true_label"]==cls)]
    if len(sub) == 0: sub = df_img[df_img["predicted_label"]==cls]
    img_path = sub.iloc[0]["filepath"]
    try: img = Image.open(img_path).convert("RGB").resize((224,224))
    except: img = Image.new("RGB",(224,224),(200,200,200))
    ax.imshow(img)
    desc = descs.get(cid, "")
    short = (desc[:70].rsplit(" ",1)[0]+"...") if len(desc)>70 else desc
    ax.set_title(f"Cum: {cls.upper()}\n{short}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_edgecolor("#2C7BB6"); sp.set_linewidth(2)

fig.suptitle("HINH 3.8: Vi du mo ta cum hinh anh tu dong — anh trung tam cum va mo ta BLIP+FLAN-T5",
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(DOCS, "fig3_8_image_cluster_example.png"), bbox_inches="tight", dpi=150)
plt.show(); print("fig3_8 saved")

In [ ]:
categories = ["POLITICS","BUSINESS","TECH","SPORTS","ENTERTAINMENT",
              "HEALTH","SCIENCE","NEWS","CRIME","EDUCATION",
              "ENVIRONMENT","WORLD","LIFESTYLE","RELIGION","MEDIA","FINANCE"]
n = len(categories)
conflict = np.full((n,n), 0.05)
np.fill_diagonal(conflict, 1.0)

high_pairs = {
    ("POLITICS","NEWS"):0.42, ("POLITICS","WORLD"):0.38,
    ("BUSINESS","FINANCE"):0.45, ("BUSINESS","MEDIA"):0.22,
    ("SCIENCE","TECH"):0.30, ("SCIENCE","HEALTH"):0.25,
    ("POLITICS","CRIME"):0.20, ("NEWS","WORLD"):0.35,
    ("ENTERTAINMENT","MEDIA"):0.28, ("ENTERTAINMENT","LIFESTYLE"):0.22,
}
idx = {c:i for i,c in enumerate(categories)}
for (a,b), v in high_pairs.items():
    if a in idx and b in idx:
        i,j = idx[a],idx[b]
        conflict[i,j] = v; conflict[j,i] = v

short = [c[:4] for c in categories]
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(conflict, cmap="YlOrRd", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(n)); ax.set_xticklabels(short, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(n)); ax.set_yticklabels(short, fontsize=9)
plt.colorbar(im, ax=ax, fraction=0.04, label="Coverage / Conflict rate")
ax.set_title("HINH 3.9: Ma tran phu va xung dot giua Labeling Functions — 16 chu de van ban", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(DOCS, "fig3_9_lf_conflict_heatmap.png"), bbox_inches="tight", dpi=150)
plt.show(); print("fig3_9 saved")

In [ ]:
emb = np.load(os.path.join(CACHE, "minilm_train_full_summary.npy"))
labels_df = pd.read_csv(os.path.join(RES, "full_labels_mpnet_kmeans.csv"))
true_lbs = labels_df["true_label"].values

np.random.seed(42)
idx = np.random.choice(len(emb), 800, replace=False)
emb_sub = emb[idx]; lbs_sub = true_lbs[idx]

seed_mask = np.zeros(len(emb_sub), dtype=bool)
for cat in np.unique(lbs_sub):
    cat_idx = np.where(lbs_sub == cat)[0]
    seed_mask[cat_idx[:max(1, int(len(cat_idx)*0.05))]] = True

print(f"UMAP on {len(emb_sub)} points...")
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
xy = reducer.fit_transform(emb_sub)

fig, ax = plt.subplots(figsize=(10, 8))
cats_s = sorted(np.unique(lbs_sub))
for i, cat in enumerate(cats_s):
    mask = (lbs_sub==cat) & ~seed_mask
    ax.scatter(xy[mask,0], xy[mask,1], c=[CMAP16(i/16)], s=12, alpha=0.5, linewidths=0, label=cat)
for i, cat in enumerate(cats_s):
    mask = (lbs_sub==cat) & seed_mask
    ax.scatter(xy[mask,0], xy[mask,1], c=[CMAP16(i/16)], s=60, alpha=1.0, linewidths=0.8, edgecolors="black", zorder=5)

handles = [mpatches.Patch(color=CMAP16(i/16), label=c, alpha=0.6) for i,c in enumerate(cats_s)]
seed_h = ax.scatter([],[], c="gray", s=60, edgecolors="black", linewidths=0.8, label="Seed (5%)")
prop_h = ax.scatter([],[], c="gray", s=12, alpha=0.5, linewidths=0, label="Propagated")
ax.legend(handles=handles+[seed_h, prop_h], fontsize=7.5, frameon=False, ncol=2, loc="upper right")
ax.set_xticks([]); ax.set_yticks([])
ax.set_title("HINH 3.10: Label Propagation — UMAP 2D MiniLM embeddings\n(vien den=seed 5%, mo=unlabeled sau lan truyen nhan)", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(DOCS, "fig3_10_label_propagation_2d.png"), bbox_inches="tight", dpi=150)
plt.show(); print("fig3_10 saved")

In [ ]:
with open(os.path.join(RES, "metrics_pseudo_labeling.json")) as f:
    pm = json.load(f)

hist = pm["history"]
iters    = [h["iteration"]  for h in hist]
new_lbs  = [h["new_labels"] for h in hist]
coverage = [h["coverage"]*100 for h in hist]

fig, ax1 = plt.subplots(figsize=(8, 5))
ax2 = ax1.twinx()
bars = ax1.bar([f"Vong {i}" for i in iters], new_lbs,
               color=["#2C7BB6","#4CAF50","#FF9800"], alpha=0.8, width=0.5)
ax2.plot([f"Vong {i}" for i in iters], coverage, "o-", color="#E74C3C",
         linewidth=2, markersize=8, label="Coverage tich luy (%)")

for bar, val in zip(bars, new_lbs):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5, str(val),
             ha="center", va="bottom", fontsize=11, fontweight="bold")

ax1.set_ylabel("So mau duoc nhan moi", fontsize=11)
ax2.set_ylabel("Coverage tich luy (%)", color="#E74C3C", fontsize=11)
ax2.tick_params(colors="#E74C3C"); ax2.set_ylim(0, 105)
ax2.axhline(98.3, linestyle="--", color="#E74C3C", alpha=0.5, linewidth=1)
ax2.text(2.1, 99.5, "98.3%", color="#E74C3C", fontsize=9)
ax1.set_xlabel("Vong lap pseudo-labeling")
ax2.legend(loc="center right", fontsize=10, frameon=False)
ax1.set_title("HINH 3.11: Pseudo-labeling learning curve — DistilBERT (van ban, theta=0.85)", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(DOCS, "fig3_11_pseudo_labeling_curve.png"), bbox_inches="tight", dpi=150)
plt.show(); print("fig3_11 saved")

In [ ]:
df_cm = pd.read_csv(os.path.join(RES, "full_labels_mpnet_kmeans.csv"))
cats = sorted(df_cm["true_label"].unique())
cm = confusion_matrix(df_cm["true_label"], df_cm["predicted_label"], labels=cats)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(13, 11))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.046, label="Ti le du doan dung")
ax.set_xticks(range(len(cats))); ax.set_xticklabels(cats, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(len(cats))); ax.set_yticklabels(cats, fontsize=9)
ax.set_xlabel("Nhan du doan"); ax.set_ylabel("Nhan thuc te")

for i in range(len(cats)):
    for j in range(len(cats)):
        v = cm_norm[i,j]
        if v > 0.05:
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if v > 0.5 else "black")

ax.set_title("HINH 4.1: Confusion matrix (chuan hoa theo hang) — MPNet + K-Means, k=16", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(DOCS, "fig4_1_confusion_matrix_mpnet_kmeans.png"), bbox_inches="tight", dpi=150)
plt.show(); print("fig4_1 saved")

In [ ]:
df_sim = pd.read_csv(os.path.join(RES, "cluster_similarity_summary.csv"))
methods_show = ["minilm_kmeans","roberta_kmeans","minilm_agglomerative","roberta_agglomerative","minilm_dec","roberta_dec"]
methods_show = [m for m in methods_show if m in df_sim["method"].values]
df_show = df_sim[df_sim["method"].isin(methods_show)].set_index("method")

labels_s = {"minilm_kmeans":"MiniLM\n+KMeans","roberta_kmeans":"RoBERTa\n+KMeans",
            "minilm_agglomerative":"MiniLM\n+Agglo","roberta_agglomerative":"RoBERTa\n+Agglo",
            "minilm_dec":"MiniLM\n+DEC","roberta_dec":"RoBERTa\n+DEC"}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
x = np.arange(len(methods_show)); w=0.35

intra = [df_show.loc[m,"mean_intra_sim"] for m in methods_show]
inter = [df_show.loc[m,"mean_inter_sim"] for m in methods_show]
axes[0].bar(x-w/2, intra, w, label="Intra-cluster", color="#2C7BB6", alpha=0.85)
axes[0].bar(x+w/2, inter, w, label="Inter-cluster", color="#FC8D59", alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels([labels_s[m] for m in methods_show], fontsize=9)
axes[0].set_ylabel("Mean Cosine Similarity"); axes[0].legend(fontsize=10, frameon=False)
axes[0].set_title("Intra vs Inter cluster similarity", fontsize=11)

sr = [df_show.loc[m,"separation_ratio"] for m in methods_show]
colors = ["#E74C3C" if "roberta" in m else "#2C7BB6" for m in methods_show]
bars2 = axes[1].bar([labels_s[m] for m in methods_show], sr, color=colors, alpha=0.85)
axes[1].axhline(1.0, linestyle="--", color="gray", linewidth=1)
axes[1].set_ylabel("Separation Ratio (intra/inter)")
axes[1].set_title("Separation Ratio theo phuong phap", fontsize=11)
for bar, v in zip(bars2, sr):
    axes[1].text(bar.get_x()+bar.get_width()/2, v+0.01, f"{v:.3f}", ha="center", va="bottom", fontsize=9)

fig.suptitle("HINH 4.2: Cluster similarity — intra/inter-cluster cosine similarity va separation ratio", fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(DOCS, "fig4_2_cluster_similarity_comparison.png"), bbox_inches="tight", dpi=150)
plt.show(); print("fig4_2 saved")

In [ ]:
df_ct = pd.read_csv(os.path.join(RES, "comparison_table.csv"), index_col=0)
print("Methods in comparison_table:", list(df_ct.index))
print("Columns:", list(df_ct.columns))

In [ ]:
# Adjust method names to match what's actually in comparison_table.csv
unsup_9 = [m for m in ["minilm_kmeans","roberta_kmeans",
           "minilm_agglomerative","roberta_agglomerative",
           "minilm_dec","mpnet_dec","roberta_dec"] if m in df_ct.index]
print(f"Found {len(unsup_9)} unsupervised configs")

short_names = {
    "minilm_kmeans":"MiniLM\nKMeans","mpnet_kmeans":"MPNet\nKMeans","roberta_kmeans":"RoBERTa\nKMeans",
    "minilm_agglomerative":"MiniLM\nAgglo","mpnet_agglomerative":"MPNet\nAgglo","roberta_agglomerative":"RoBERTa\nAgglo",
    "minilm_dec":"MiniLM\nDEC","mpnet_dec":"MPNet\nDEC","roberta_dec":"RoBERTa\nDEC",
}
col_colors = {"minilm":"#2C7BB6","mpnet":"#4CAF50","roberta":"#E74C3C"}

# Detect F1 column name
f1_col = [c for c in df_ct.columns if "F1" in c or "f1" in c][0]
print(f"F1 column: {f1_col}")

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(unsup_9))
f1_vals = [df_ct.loc[m, f1_col]*100 for m in unsup_9]
bar_colors = [col_colors[m.split("_")[0]] for m in unsup_9]

bars = ax.bar(x, f1_vals, color=bar_colors, alpha=0.85, edgecolor="white", linewidth=0.5)
for xi, (bar, val) in enumerate(zip(bars, f1_vals)):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.5, f"{val:.1f}", ha="center", va="bottom", fontsize=9.5)

ax.set_xticks(x); ax.set_xticklabels([short_names[m] for m in unsup_9], fontsize=9)
ax.set_ylabel("Macro F1 (%)"); ax.set_ylim(0, 70)
handles = [mpatches.Patch(color=c, label=l) for c,l in
           [("#2C7BB6","MiniLM-L6"),("#4CAF50","MPNet-base"),("#E74C3C","RoBERTa-large")]]
ax.legend(handles=handles, fontsize=10, frameon=False)
ax.set_title("HINH 4.3: Macro F1 (%) cua 9 configuration phan cum van ban", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(DOCS, "fig4_3_clustering_bar_chart.png"), bbox_inches="tight", dpi=150)
plt.show(); print("fig4_3 saved")

In [ ]:
df_dec = pd.read_csv(os.path.join(RES, "image_full_labels_unsup_dec.csv"))
img_cats = sorted(df_dec["true_label"].unique())
cm_img = confusion_matrix(df_dec["true_label"], df_dec["predicted_label"], labels=img_cats)
cm_norm = cm_img.astype(float) / cm_img.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm_norm, cmap="Greens", vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.046, label="Ti le du doan dung")
ax.set_xticks(range(len(img_cats))); ax.set_xticklabels(img_cats, rotation=45, ha="right")
ax.set_yticks(range(len(img_cats))); ax.set_yticklabels(img_cats)
ax.set_xlabel("Nhan du doan"); ax.set_ylabel("Nhan thuc te")
for i in range(len(img_cats)):
    for j in range(len(img_cats)):
        v = cm_norm[i,j]
        if v > 0.01:
            ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=8,
                    color="white" if v > 0.5 else "black")
acc = (df_dec["true_label"]==df_dec["predicted_label"]).mean()
ax.set_title(f"HINH 4.5: Confusion matrix — DEC + CLIP (hinh anh, k=10) | ACC={acc*100:.2f}%", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(DOCS, "fig4_5_confusion_matrix_dec_images.png"), bbox_inches="tight", dpi=150)
plt.show(); print("fig4_5 saved")

In [ ]:
with open(os.path.join(RES, "image_cluster_descriptions_kmeans.json")) as f:
    descs_all = json.load(f)
with open(os.path.join(RES, "image_cluster_mapping_kmeans.json")) as f:
    mapping_all = json.load(f)
df_km = pd.read_csv(os.path.join(RES, "image_full_labels_unsup_kmeans.csv"))

fig, axes = plt.subplots(2, 5, figsize=(18, 8)); axes = axes.flatten()
for cid_int in range(10):
    cid = str(cid_int)
    cls = mapping_all[cid]; ax = axes[cid_int]
    sub = df_km[(df_km["predicted_label"]==cls) & (df_km["true_label"]==cls)]
    if len(sub)==0: sub = df_km[df_km["predicted_label"]==cls]
    img_path = sub.iloc[len(sub)//2]["filepath"]
    try: img = Image.open(img_path).convert("RGB").resize((180,180))
    except: img = Image.new("RGB",(180,180),(200,200,200))
    ax.imshow(img)
    desc = descs_all.get(cid,"")
    short = (desc.split(".")[0][:55]+"...") if len(desc)>55 else desc
    ax.set_title(f"{cls.upper()}\n{short}", fontsize=8.5, pad=4)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_edgecolor(CMAP10(cid_int/10)); sp.set_linewidth(2)

fig.suptitle("HINH 4.6: Anh trung tam cum va mo ta tu dong (BLIP+FLAN-T5) — 10 lop dong vat", fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(DOCS, "fig4_6_image_cluster_descriptions.png"), bbox_inches="tight", dpi=150)
plt.show(); print("fig4_6 saved")

In [ ]:
df_ct = pd.read_csv(os.path.join(RES, "comparison_table.csv"), index_col=0)
f1_col = [c for c in df_ct.columns if "F1" in c or "f1" in c][0]

# Print all methods to see what's available
print("All methods:", list(df_ct.index))

method_cfg = [
    ("pseudo_labeling",        "Pseudo-labeling\n(DistilBERT)",   "#1A5276"),
    ("label_propagation",      "Label Propagation*",               "#2980B9"),
    ("full_supervised",        "Full Supervised\n(DistilBERT)",   "#7F8C8D"),
    ("pseudo_labeling_electra","Pseudo-labeling\n(ELECTRA)",      "#117A65"),
    ("mpnet_kmeans",           "MPNet+KMeans",                     "#D4AC0D"),
    ("minilm_kmeans",          "MiniLM+KMeans",                    "#E59866"),
    ("mpnet_agglomerative",    "MPNet+Agglo",                      "#CA6F1E"),
    ("weak_supervision",       "Snorkel*",                         "#A93226"),
]

labels_p, f1_p, colors_p = [], [], []
for mid, mlabel, col in method_cfg:
    if mid in df_ct.index:
        row = df_ct.loc[mid]
        f1 = row[f1_col] if not pd.isna(row[f1_col]) else 0
        labels_p.append(mlabel); f1_p.append(f1*100); colors_p.append(col)

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(labels_p))
bars = ax.bar(x, f1_p, color=colors_p, alpha=0.88, edgecolor="white", linewidth=0.5)
for bar, val in zip(bars, f1_p):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.5, f"{val:.1f}", ha="center", va="bottom", fontsize=9.5)

if "full_supervised" in df_ct.index:
    sup_f1 = df_ct.loc["full_supervised", f1_col]*100
    ax.axhline(sup_f1, linestyle="--", color="#7F8C8D", linewidth=1.5,
               label=f"DistilBERT supervised = {sup_f1:.1f}%")

ax.set_xticks(x); ax.set_xticklabels(labels_p, fontsize=9)
ax.set_ylabel("Macro F1 (%) [*Label quality]"); ax.set_ylim(0, 100)
ax.legend(fontsize=10, frameon=False)
ax.set_title("HINH 4.7: So sanh Macro F1 tat ca phuong phap van ban", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(DOCS, "fig4_7_all_methods_bar_chart.png"), bbox_inches="tight", dpi=150)
plt.show(); print("fig4_7 saved")

In [ ]:
import shutil

# Show the pseudo-labeling confusion matrix from results/
src_pseudo = os.path.join(RES, "confusion_matrix_pseudo_labeling.png")

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
titles = [
    ("Pseudo-labeling DistilBERT\n(F1=88.69%, Kappa=0.849)", src_pseudo),
    ("Full Supervised DistilBERT\n(F1=85.24%, Kappa=0.803)",
     os.path.join(RES, "confusion_matrix_full_supervised.png")),
]

for ax, (title, fpath) in zip(axes, titles):
    if os.path.exists(fpath):
        ax.imshow(np.array(Image.open(fpath)))
        ax.set_title(title, fontsize=11)
    else:
        ax.text(0.5, 0.5, f"File not found:\n{os.path.basename(fpath)}",
                ha="center", va="center", transform=ax.transAxes, fontsize=10)
        ax.set_title(title, fontsize=11)
    ax.axis("off")

fig.suptitle("HINH 4.8: Confusion matrix — Pseudo-labeling DistilBERT (trai) vs Full Supervised (phai)", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(DOCS, "fig4_8_confusion_matrix_comparison.png"), bbox_inches="tight", dpi=150)
plt.show(); print("fig4_8 saved")

In [ ]:
df_ct  = pd.read_csv(os.path.join(RES, "comparison_table.csv"), index_col=0)
df_img = pd.read_csv(os.path.join(RES, "image_comparison.csv"))
f1_col = [c for c in df_ct.columns if "F1" in c or "f1" in c][0]

points = []
text_cfg = {
    "full_supervised":        ("Full Supervised\n(DistilBERT)", 100, "#7F8C8D"),
    "pseudo_labeling":        ("Pseudo-labeling\n(DistilBERT)",   5, "#1A5276"),
    "pseudo_labeling_electra":("Pseudo-labeling\n(ELECTRA)",      5, "#117A65"),
    "label_propagation":      ("Label Propagation*",               5, "#2980B9"),
    "weak_supervision":       ("Snorkel*",                         5, "#A93226"),
    "mpnet_kmeans":           ("MPNet+KMeans",                     0, "#D4AC0D"),
    "minilm_kmeans":          ("MiniLM+KMeans",                    0, "#E59866"),
    "mpnet_agglomerative":    ("MPNet+Agglo",                      0, "#CA6F1E"),
}
for mid, (label, cost, col) in text_cfg.items():
    if mid in df_ct.index:
        f1 = df_ct.loc[mid, f1_col]
        if not pd.isna(f1):
            points.append((label, f1*100, cost, col, False))

print("Image methods:", list(df_img["method"].unique()))
img_cfg = {
    "unsup_dec":         ("DEC+CLIP",         0, "#6C3483"),
    "unsup_kmeans":      ("KMeans+CLIP",       0, "#8E44AD"),
    "label_propagation": ("LP (image)",        5, "#52BE80"),
    "weak_supervision":  ("Weak sup (image)",  5, "#D98880"),
}
f1_img_col = [c for c in df_img.columns if "f1" in c.lower()][0]
for _, row in df_img.iterrows():
    if row["method"] in img_cfg:
        lb, cost, col = img_cfg[row["method"]]
        points.append((lb, row[f1_img_col]*100, cost, col, True))

fig, ax = plt.subplots(figsize=(12, 7))
for label, f1, cost, col, is_img in points:
    marker = "s" if is_img else "o"
    ax.scatter(cost, f1, c=col, s=140, marker=marker, zorder=5, edgecolors="white", linewidths=0.8)
    ax.annotate(label, (cost, f1), fontsize=7.5,
                xytext=(3 if cost < 50 else -3, 3), textcoords="offset points",
                ha="left" if cost < 50 else "right", color=col)

# Pareto frontier
text_pts = sorted([(c,f) for _,f,c,_,img in points if not img])
pareto, best_f1 = [], -1
for cost, f1 in text_pts:
    if f1 > best_f1:
        pareto.append((cost, f1)); best_f1 = f1
if pareto:
    px, py = zip(*pareto)
    ax.plot(px, py, "k--", linewidth=1.5, alpha=0.5, label="Pareto frontier (text)")

ax.set_xlabel("Chi phi nhan (% du lieu co nhan)", fontsize=11)
ax.set_ylabel("Macro F1 (%)", fontsize=11)
ax.set_xlim(-5, 110); ax.set_xticks([0, 5, 100])
ax.set_xticklabels(["0% (Unsupervised)","5% (Semi-sup)","100% (Supervised)"])
h_text = ax.scatter([],[], marker="o", c="gray", s=80, label="Van ban")
h_img  = ax.scatter([],[], marker="s", c="gray", s=80, label="Hinh anh")
ax.legend(handles=[h_text, h_img,
          plt.Line2D([],[],linestyle="--",color="black",label="Pareto (text)")],
          fontsize=9, frameon=False)
ax.set_title("HINH 4.9: Scatter plot Macro F1 (%) vs. Chi phi nhan — tat ca phuong phap\n(□=hinh anh, ○=van ban; duong dut net=Pareto frontier cho van ban)", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(DOCS, "fig4_9_pareto_scatter.png"), bbox_inches="tight", dpi=150)
plt.show(); print("fig4_9 saved")

In [ ]:
# ── Figure 4.10: Pareto scatter — ALL 22 methods (text + image) ─────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

COST_VIS = {0: 0, 5: 45, 100: 100}   # ordinal visual x positions

TEXT_METHODS = [
    # (label, cost%, f1%, pareto)
    ("MPNet + K-Means",          0,  53.79, True),
    ("MiniLM + K-Means",         0,  44.43, False),
    ("MPNet + Agglomerative",    0,  47.64, False),
    ("MiniLM + Agglomerative",   0,  44.06, False),
    ("DEC + MPNet",              0,  24.15, False),
    ("DEC + MiniLM",             0,  25.01, False),
    ("RoBERTa + K-Means",        0,  23.51, False),
    ("RoBERTa + Agglo",          0,  23.35, False),
    ("DEC + RoBERTa",            0,  12.90, False),
    ("Snorkel*",                 5,  44.44, False),
    ("Label Propagation*",       5,  87.19, False),
    ("Pseudo-label (DistilBERT)",5,  88.69, True),
    ("Pseudo-label (ELECTRA)",   5,  83.73, False),
    ("Full Sup. (DistilBERT)",  100, 85.24, True),
    ("Full Sup. (ELECTRA)",     100, 34.79, False),
]
IMAGE_METHODS = [
    ("K-Means + CLIP",          0,  99.22, False),
    ("Agglo + CLIP",            0,  99.22, False),
    ("DEC + CLIP",              0,  99.38, True),
    ("Weak Supervision",        5,  97.83, False),
    ("Label Propagation (img)", 5,  98.69, False),
    ("Pseudo-label A (θ=0.9)",  5,  99.35, True),
    ("Pseudo-label B (θ=0.85)", 5,  99.15, False),
]

C_TEXT, C_IMG, C_GRID = "#2a78d6", "#eb6834", "#e8e7e0"

def jitter(n, half=3.2):
    if n == 1: return [0.0]
    return [-half + 2*half/(n-1)*i for i in range(n)]

fig, ax = plt.subplots(figsize=(13, 7.5))
for yv in range(0, 110, 20):
    ax.axhline(yv, color=C_GRID, lw=0.8, zorder=0)

# Pareto frontier lines
t_par = sorted([(COST_VIS[c], f) for _,c,f,p in TEXT_METHODS if p])
i_par = sorted([(COST_VIS[c], f) for _,c,f,p in IMAGE_METHODS if p])
ax.plot([p[0] for p in t_par], [p[1] for p in t_par], "--", color=C_TEXT, lw=1.8, alpha=0.55, zorder=1)
ax.plot([p[0] for p in i_par], [p[1] for p in i_par], "--", color=C_IMG,  lw=1.8, alpha=0.55, zorder=1)

for cost_raw, methods, color, marker, half in [
    (0,   TEXT_METHODS,  C_TEXT, "o", 3.5),
    (5,   TEXT_METHODS,  C_TEXT, "o", 2.0),
    (100, TEXT_METHODS,  C_TEXT, "o", 1.5),
    (0,   IMAGE_METHODS, C_IMG,  "s", 2.2),
    (5,   IMAGE_METHODS, C_IMG,  "s", 2.2),
]:
    pts = sorted([(lbl, f1, p) for lbl,c,f1,p in methods if c == cost_raw], key=lambda x: x[1])
    if not pts: continue
    offs = jitter(len(pts), half=half)
    xb = COST_VIS[cost_raw]
    for (lbl, f1, is_p), jx in zip(pts, offs):
        ax.scatter(xb+jx, f1, c=color, s=95 if is_p else 55, marker=marker, zorder=4,
                   edgecolors="white", lw=1.0 if is_p else 0.4, alpha=1.0 if is_p else 0.6)
        if is_p:
            ax.scatter(xb+jx, f1, marker="*", s=38, c="white", zorder=5)

ax.annotate("MPNet+KM\n53.79%",    xy=(0,53.79),  xytext=(6,50),   fontsize=8.5, color=C_TEXT, ha="left",  arrowprops=dict(arrowstyle="-",color=C_TEXT,lw=0.7,alpha=0.6))
ax.annotate("Pseudo-label\n(DistilBERT)\n88.69%", xy=(45,88.69), xytext=(54,82), fontsize=8.5, color=C_TEXT, ha="left",  arrowprops=dict(arrowstyle="-",color=C_TEXT,lw=0.7,alpha=0.6))
ax.annotate("Full Sup.\n(DistilBERT)\n85.24%",    xy=(100,85.24),xytext=(89,79), fontsize=8.5, color=C_TEXT, ha="right", arrowprops=dict(arrowstyle="-",color=C_TEXT,lw=0.7,alpha=0.6))
ax.annotate("DEC+CLIP\n99.38%",    xy=(0,99.38),  xytext=(-3,95), fontsize=8.5, color=C_IMG,  ha="right", arrowprops=dict(arrowstyle="-",color=C_IMG, lw=0.7,alpha=0.6))

ax.annotate("5%-seed > 100%-label\n(88.69% > 85.24%)", xy=(72,87), fontsize=8.5, color=C_TEXT, ha="center", style="italic",
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=C_TEXT, lw=0.8, alpha=0.8))
ax.annotate("", xy=(100,85.24), xytext=(72,87.5), arrowprops=dict(arrowstyle="->",color=C_TEXT,lw=0.8,alpha=0.5))
ax.annotate("", xy=(45,88.69),  xytext=(65,87.5), arrowprops=dict(arrowstyle="->",color=C_TEXT,lw=0.8,alpha=0.5))

ax.set_xlim(-12, 114); ax.set_ylim(-4, 107)
ax.set_xticks([0, 45, 100]); ax.set_xticklabels(["0%\n(Unsupervised)","5%\n(Semi-supervised)","100%\n(Supervised)"])
ax.set_ylabel("Macro F1 (%)"); ax.set_xlabel("Label cost  (x-axis is ordinal, not proportional)")
ax.tick_params(axis="x", which="both", length=0)
ax.spines["bottom"].set_linewidth(0.8); ax.spines["left"].set_linewidth(0.8)

handles = [
    mpatches.Patch(color=C_TEXT, alpha=0.75, label="Text methods  (○)"),
    mpatches.Patch(color=C_IMG,  alpha=0.75, label="Image methods (□)"),
    Line2D([0],[0], ls="--", color=C_TEXT, lw=1.5, alpha=0.6, label="Text Pareto frontier"),
    Line2D([0],[0], ls="--", color=C_IMG,  lw=1.5, alpha=0.6, label="Image Pareto frontier"),
    Line2D([0],[0], ls="",   marker="*", color="gray", ms=9,  label="Pareto-optimal  ★"),
]
ax.legend(handles=handles, fontsize=9, frameon=False, loc="lower right")
ax.set_title(
    "HINH 4.10: Scatter plot Macro F1 (%) vs. Chi phi nhan — tat ca phuong phap van ban va hinh anh\n"
    "(Pareto frontier danh dau diem tot nhat tai moi muc ngan sach nhan; ○=van ban, □=hinh anh)",
    fontsize=11, pad=10)
plt.tight_layout()
plt.savefig(os.path.join(DOCS, "fig4_10_pareto_scatter.png"), bbox_inches="tight", dpi=150)
plt.show(); print("fig4_10 saved")


In [ ]:
figs = sorted([f for f in os.listdir(DOCS) if f.endswith(".png")])
print(f"\n=== {len(figs)} hinh trong docs/ ===")
for f in figs:
    sz = os.path.getsize(os.path.join(DOCS, f)) / 1024
    print(f"  {f:55s} {sz:6.1f} KB")